# 06 Baseline audit (Prompt 10)

**Needs a GPU**: Runtime → Change runtime type → **T4 GPU**. First upload into the Drive project folder: `src/audit.py`, `src/generate.py` (small change: the audit can reach the loaded model), `scripts/audit_baseline.py`, `tests/test_audit.py`, `notebooks/06_baseline_audit.ipynb`.

Then Runtime → **Run all** (about 35 minutes in total). **Keep this tab open.** If Colab disconnects, reconnect and Run all again: cells 5 and 6 continue where they stopped. Only **development** items are used. Copy all the text output back into the chat (long outputs: use the scroll area's text, not a screenshot).

In [ ]:
# 1. Drive, project folder, libraries
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/atr_colab/AITR_Lab02_MCL2026002_PromptStability
!pip install -q "transformers>=4.37.0" accelerate bitsandbytes pyyaml scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Unit tests: audit module, and the generation tests again (generate.py changed)
!python tests/test_audit.py
!python tests/test_generate.py 2>&1 | tail -1
!python tests/test_evaluate.py 2>&1 | tail -1

In [ ]:
# 3. Static audit (no GPU work): records, split, decoding, re-scoring, 'maybe' pattern, official evaluation.py
!python scripts/audit_baseline.py --check static

In [ ]:
# 4. Reproducibility: rerun 20 pilot items, compare with the saved dev_b0 text (about 4 minutes)
!python scripts/audit_baseline.py --check repro 2>&1 | grep -v -E 'Downloading|Fetching|Loading checkpoint|it/s\]'

In [ ]:
# 5. Answer-format diagnostic: 100 dev items x 2 formats (about 12 minutes, resumable)
!python scripts/audit_baseline.py --check format 2>&1 | grep -v -E 'Downloading|Fetching|Loading checkpoint|it/s\]'

In [ ]:
# 6. Quantisation diagnostic: the same 100 items, P0 in 8-bit (about 15-20 minutes, resumable)
# Run this in a fresh process (it is: each ! command is a new process), so the 4-bit model is not in memory.
!python scripts/audit_baseline.py --check quant8 2>&1 | grep -v -E 'Downloading|Fetching|Loading checkpoint|it/s\]'

In [ ]:
# 7. Files written
!ls -la results/audit